# Nearest neighbors: making similarity operational

**Question.** If two wine samples are chemically similar, do their recorded outcomes tend to be similar?

Nearest-neighbor methods turn a distance function into a prediction rule. That makes units, scaling, irrelevant dimensions, duplicated records, data density, and query latency part of the scientific contract—not preprocessing trivia.

**What this example models.** The applied claim ‘similar cases have similar outcomes’ is made testable through a distance, neighborhood, validation design, and local failure inspection.

**Your project version.** Define what similarity should mean in the domain, implement and test the neighbor rule, justify scaling and distance choices, compare neighborhood sizes without contaminating the final test set, inspect local explanations, and measure prediction cost.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from dsm_project.config import load_config
from dsm_project.data import load_project_dataset
from dsm_project.experiments import fit_supervised_candidate

config = load_config("configs/example.toml")
dataset = load_project_dataset(config)
classification = dataset.tasks["classification"]
frame = dataset.observations

scale_audit = frame[list(classification.feature_columns)].agg(["min", "max", "std"]).T
scale_audit["range"] = scale_audit["max"] - scale_audit["min"]
display(scale_audit.sort_values("range", ascending=False))

## Distance, neighborhood, and prediction

For standardized vectors, Euclidean distance is

$$
d(x,x')=\sqrt{\sum_{j=1}^{p}(x_j-x'_j)^2}.
$$

A $k$-nearest-neighbor classifier finds the $k$ training vectors with smallest distance and aggregates their labels, commonly by a vote. A regressor aggregates targets, commonly by a mean. Small $k$ can follow noise; large $k$ can wash out local structure.

The table above shows why raw Euclidean distance would be dominated by large-range columns. Standardization is fitted using training rows inside the reference pipeline. This prevents holdout statistics from leaking into training.

In [ ]:
classification_result = fit_supervised_candidate(
    dataset,
    config,
    task_name="classification",
    model_name="neighbors",
)
regression_result = fit_supervised_candidate(
    dataset,
    config,
    task_name="regression",
    model_name="neighbors",
)
print(classification_result.model_name, classification_result.metric_name, classification_result.metric_value)
print(regression_result.model_name, regression_result.metric_name, regression_result.metric_value)

## Inspect one local explanation

A neighbor model can expose which training rows influenced a query. That is useful, but proximity is not causality and a nearby record can still be a poor analogue. We inspect distances in the fitted standardized space and preserve the distinction between row identifiers and model features.

In [ ]:
pipeline = classification_result.estimator
scaled_query = pipeline.named_steps["standardscaler"].transform(
    classification_result.holdout_features.iloc[[0]]
)
distances, positions = pipeline.named_steps["kneighborsclassifier"].kneighbors(
    scaled_query,
    n_neighbors=7,
)

neighbor_evidence = pd.DataFrame(
    {
        "training_position": positions[0],
        "standardized_distance": distances[0],
    }
)
display(neighbor_evidence)

plt.plot(range(1, 8), distances[0], marker="o", color="#7570b3")
plt.xlabel("neighbor rank")
plt.ylabel("standardized Euclidean distance")
plt.title("Local density around one held-out query")
plt.tight_layout();

## Conclusion, limits, and student implementation

The classification and regression calls share a family but use different aggregation rules and metrics. A useful project analysis varies $k$, distance, weighting, and feature representation inside a validation procedure; checks duplicated and near-duplicated records; studies performance in sparse regions; and measures query latency as training data grow.

A student implementation should test exact distances, tie behavior, self-neighbor handling, invalid $k$, classification voting, regression aggregation, and deterministic results. Its notebook should explain why the chosen notion of similarity is scientifically meaningful for the project's observation unit.